## 1. Import Libraries

In [ ]:
import os
import json
import copy
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision.transforms as T
from torchvision import models

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix
)

## 2. Reproducibility, Paths, and Training Configuration

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

# Set this to the location of your downloaded China-Fundus-CIMT dataset
BASE_DIR = Path("/path/to/China_CIMT_dataset")

JSON_PATH = BASE_DIR / "data_info.json"

# Existing final preprocessed PNG images
IMG_DIR = BASE_DIR / "Fundus_green_CLAHE_minmax"

# Cached tensors will be saved here
CACHE_DIR = BASE_DIR / "EfficientNetB0_tensor_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

OUT_DIR = BASE_DIR / "EfficientNetB0_FundusOnly"
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 30
PATIENCE = 7
LR = 1e-4
WEIGHT_DECAY = 1e-4

DEVICE: cuda


## 3. Load and Validate Dataset Metadata

In [3]:
KEY_LABEL = "label"
KEY_GROUP = "group"
KEY_LEFT  = "left_eye"
KEY_RIGHT = "right_eye"

def build_df_from_json(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)

    rows = []

    for pid, info in data.items():
        label = int(info[KEY_LABEL])
        group = int(info[KEY_GROUP])

        if info.get(KEY_LEFT):
            rows.append({
                "filename": info[KEY_LEFT],
                "label": label,
                "group": group
            })

        if info.get(KEY_RIGHT):
            rows.append({
                "filename": info[KEY_RIGHT],
                "label": label,
                "group": group
            })

    df = pd.DataFrame(rows)

    if df.empty:
        raise RuntimeError("df_all is empty. Check JSON file.")

    return df

df_all = build_df_from_json(JSON_PATH)

print("Total images:", len(df_all))
print(df_all.head())
print(df_all.groupby(["group", "label"]).size())

Total images: 5806
        filename  label  group
0  2491006_L.png      0      1
1  2491006_R.png      0      1
2  3730004_L.png      1      1
3  3730004_R.png      1      1
4  3730006_L.png      1      1
group  label
1      0        1398
       1        3808
2      0         200
       1         200
3      0         100
       1         100
dtype: int64


## 4. Dataset Split Summary

In [4]:
train_df = df_all[df_all["group"] == 1].copy()
val_df   = df_all[df_all["group"] == 2].copy()
test_df  = df_all[df_all["group"] == 3].copy()

print("Train:", train_df.shape, train_df["label"].value_counts().to_dict())
print("Val  :", val_df.shape, val_df["label"].value_counts().to_dict())
print("Test :", test_df.shape, test_df["label"].value_counts().to_dict())

Train: (5206, 3) {1: 3808, 0: 1398}
Val  : (400, 3) {1: 200, 0: 200}
Test : (200, 3) {1: 100, 0: 100}


## 5. Tensor Cache Generation

In [5]:
def cache_one_image(filename):
    base = Path(filename).stem
    out_path = CACHE_DIR / f"{base}.pt"

    if out_path.exists():
        return "skipped"

    img_path = IMG_DIR / filename

    if not img_path.exists():
        raise FileNotFoundError(img_path)

    img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)

    if img is None:
        raise RuntimeError(f"Could not read image: {img_path}")

    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)


    img = np.stack([img, img, img], axis=0)  # C,H,W

    img = torch.tensor(img, dtype=torch.float32) / 255.0

    torch.save(img, out_path)

    return "computed"


ok, skipped, fail = 0, 0, 0

for fn in tqdm(df_all["filename"].tolist(), desc="CACHE_TENSORS"):
    try:
        status = cache_one_image(fn)

        if status == "skipped":
            skipped += 1
        else:
            ok += 1

    except Exception as e:
        fail += 1
        print("Cache failed:", fn, e)

print("Tensor cache done.")
print("computed =", ok, "skipped =", skipped, "failed =", fail)

CACHE_TENSORS: 100%|██████████| 5806/5806 [00:26<00:00, 221.00it/s]

Tensor cache done.
computed = 5806 skipped = 0 failed = 0


## 6. Dataset Loader

In [5]:
class FundusTensorDataset(Dataset):
    def __init__(self, df, cache_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.cache_dir = Path(cache_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        filename = row["filename"]
        label = int(row["label"])

        base = Path(filename).stem
        tensor_path = self.cache_dir / f"{base}.pt"

        if not tensor_path.exists():
            raise FileNotFoundError(tensor_path)

        img = torch.load(tensor_path)  # shape: 3,H,W

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.long)

## 7. Data Augmentation and Normalization

In [6]:
train_tfms = T.Compose([
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=10),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_tfms = T.Compose([
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## 8. Dataset and DataLoader Preparation

In [7]:
train_dataset = FundusTensorDataset(train_df, CACHE_DIR, transform=train_tfms)
val_dataset   = FundusTensorDataset(val_df, CACHE_DIR, transform=eval_tfms)
test_dataset  = FundusTensorDataset(test_df, CACHE_DIR, transform=eval_tfms)

train_labels = train_df["label"].values

class_counts = np.bincount(train_labels)
class_weights_np = 1.0 / class_counts
sample_weights = class_weights_np[train_labels]

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

class_weights = torch.tensor(
    class_weights_np / class_weights_np.sum() * 2,
    dtype=torch.float32
).to(DEVICE)

print("Class counts:", class_counts)
print("Class weights:", class_weights)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=0,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

Class counts: [1398 3808]
Class weights: tensor([1.4629, 0.5371], device='cuda:0')


## 9. Model Initialization and Training Configuration

In [8]:
from torchvision import models

model = models.efficientnet_b0(
    weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1
)

in_features = model.classifier[1].in_features

model.classifier[1] = nn.Linear(
    in_features,
    2
)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=2, bias=True)
)


## 10. Model Evaluation

In [9]:
def evaluate_model(model, loader):
    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []

    total_loss = 0.0

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            preds = torch.argmax(outputs, dim=1)

            total_loss += loss.item() * imgs.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)

    return {
        "loss": total_loss / len(loader.dataset),
        "acc": accuracy_score(y_true, y_pred),
        "prec": precision_score(y_true, y_pred, zero_division=0),
        "rec": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, y_prob),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "cm": confusion_matrix(y_true, y_pred)
    }

## 11. Model Training

In [11]:
best_val_f1 = -1.0
best_state = None
patience_counter = 0

history = []

for epoch in range(1, EPOCHS + 1):

    model.train()
    train_loss = 0.0

    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):

        imgs = imgs.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(imgs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)

    train_loss = train_loss / len(train_loader.dataset)

    val_metrics = evaluate_model(model, val_loader)

    scheduler.step(val_metrics["f1"])

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["acc"],
        "val_prec": val_metrics["prec"],
        "val_rec": val_metrics["rec"],
        "val_f1": val_metrics["f1"],
        "val_auc": val_metrics["auc"],
        "val_mcc": val_metrics["mcc"]
    }

    history.append(row)

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} | "
        f"val_acc={val_metrics['acc']:.4f} | "
        f"val_prec={val_metrics['prec']:.4f} | "
        f"val_rec={val_metrics['rec']:.4f} | "
        f"val_f1={val_metrics['f1']:.4f} | "
        f"val_auc={val_metrics['auc']:.4f} | "
        f"val_mcc={val_metrics['mcc']:.4f}"
    )

    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        best_state = copy.deepcopy(model.state_dict())
        patience_counter = 0

        torch.save(best_state,OUT_DIR / "best_efficientnet_b0.pt")
        print("Best model updated.")

    else:
        patience_counter += 1
        print("No improvement. Patience:", patience_counter, "/", PATIENCE)

    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

history_df = pd.DataFrame(history)
history_df.to_csv(OUT_DIR / "training_history.csv", index=False)

print("Best validation F1:", best_val_f1)

Epoch 1/30:   0%|          | 0/163 [00:00<?, ?it/s]C:\Users\Shri\AppData\Local\Temp\ipykernel_15048\524562542.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  img = torc

Epoch 01 | train_loss=0.4881 | val_acc=0.7275 | val_prec=0.7542 | val_rec=0.6750 | val_f1=0.7124 | val_auc=0.8204 | val_mcc=0.4575
Best model updated.


Epoch 2/30:   0%|          | 0/163 [00:00<?, ?it/s]C:\Users\Shri\AppData\Local\Temp\ipykernel_15048\524562542.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  img = torc

Epoch 02 | train_loss=0.4243 | val_acc=0.7450 | val_prec=0.7634 | val_rec=0.7100 | val_f1=0.7358 | val_auc=0.8081 | val_mcc=0.4912
Best model updated.


Epoch 3/30:   0%|          | 0/163 [00:00<?, ?it/s]C:\Users\Shri\AppData\Local\Temp\ipykernel_15048\524562542.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  img = torc

...

Epoch 09 | train_loss=0.1707 | val_acc=0.6950 | val_prec=0.6423 | val_rec=0.8800 | val_f1=0.7426 | val_auc=0.7827 | val_mcc=0.4198
No improvement. Patience: 5 / 7


Epoch 10/30:   0%|          | 0/163 [00:00<?, ?it/s]C:\Users\Shri\AppData\Local\Temp\ipykernel_15048\524562542.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  img = tor

Epoch 10 | train_loss=0.1511 | val_acc=0.6850 | val_prec=0.6360 | val_rec=0.8650 | val_f1=0.7331 | val_auc=0.7841 | val_mcc=0.3966
No improvement. Patience: 6 / 7


Epoch 11/30:   0%|          | 0/163 [00:00<?, ?it/s]C:\Users\Shri\AppData\Local\Temp\ipykernel_15048\524562542.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  img = tor

Epoch 11 | train_loss=0.1437 | val_acc=0.6525 | val_prec=0.6055 | val_rec=0.8750 | val_f1=0.7157 | val_auc=0.7884 | val_mcc=0.3406
No improvement. Patience: 7 / 7
Early stopping triggered.
Best validation F1: 0.7428571428571429


## 12. Test Set Evaluation

In [13]:
best_model_path = OUT_DIR / "best_efficientnet_b0.pt"

model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

test_metrics = evaluate_model(model, test_loader)

print("\n===== EfficientNet-B0 Fundus-Only Test Results =====")
print("Accuracy :", round(test_metrics["acc"], 4))
print("Precision:", round(test_metrics["prec"], 4))
print("Recall   :", round(test_metrics["rec"], 4))
print("F1-score :", round(test_metrics["f1"], 4))
print("AUC      :", round(test_metrics["auc"], 4))
print("MCC      :", round(test_metrics["mcc"], 4))

print("\nConfusion Matrix:")
print(test_metrics["cm"])

test_result = pd.DataFrame([{
    "model": "EfficientNet-B0",
    "input": "Fundus image only",
    "split": "Official JSON group split",
    "preprocessing": "Green channel + CLAHE + MinMax",
    "test_acc": test_metrics["acc"],
    "test_prec": test_metrics["prec"],
    "test_rec": test_metrics["rec"],
    "test_f1": test_metrics["f1"],
    "test_auc": test_metrics["auc"],
    "test_mcc": test_metrics["mcc"],
    "tn": test_metrics["cm"][0, 0],
    "fp": test_metrics["cm"][0, 1],
    "fn": test_metrics["cm"][1, 0],
    "tp": test_metrics["cm"][1, 1],
}])

test_result.to_csv(OUT_DIR / "efficientnet_b0_test_results.csv",index=False)


display(test_result)

C:\Users\Shri\AppData\Local\Temp\ipykernel_15048\990371278.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path, map_location


===== EfficientNet-B0 Fundus-Only Test Results =====
Accuracy : 0.78
Precision: 0.7692
Recall   : 0.8
F1-score : 0.7843
AUC      : 0.8403
MCC      : 0.5604

Confusion Matrix:
[[76 24]
 [20 80]]


,model,input,split,preprocessing,test_acc,test_prec,test_rec,test_f1,test_auc,test_mcc,tn,fp,fn,tp
0,EfficientNet-B0,Fundus image only,Official JSON group split,Green channel + CLAHE + MinMax,0.78,0.769231,0.8,0.784314,0.8403,0.560449,76,24,20,80
